# Research Question
Can historical customer ratings and book characteristics help predict which books a customer is likely to rate positively, supporting more personalized recommendations?

In [ ]:
import json
import pandas as pd

# Load the previously merged reviews and metadata
books_combined_final = pd.read_csv(
    "books_merged_preclean.csv.gz",
    compression="gzip",
    dtype={
        "user_id": "string",
        "parent_asin": "string",
        "asin": "string",
        "price": "string"
    },
    parse_dates=["timestamp"],
    keep_default_na=False,
    na_values={
        "price": [""],
        "main_category": [""]
    }
)

# Restore nested data structures
for column in ["images", "categories"]:
    books_combined_final[column] = (
        books_combined_final[column].apply(
            lambda value: (
                json.loads(value)
                if isinstance(value, str)
                else []
            )
        )
    )

# Restore original merge indicator type
books_combined_final["_merge"] = pd.Categorical(
    books_combined_final["_merge"],
    categories=["left_only", "right_only", "both"]
)

# Verify dataset
print("Loaded dataset:", books_combined_final.shape)

print(
    "Categories restored:",
    isinstance(books_combined_final["categories"].iloc[0], list)
)

print("Original review count:", len(books_combined_final))

In [ ]:
# Inspect combined dataset structure
books_combined_final.info()

# Missing-value summary
missing_summary = pd.DataFrame({
    "Missing Count": books_combined_final.isna().sum(),
    "Missing %": (
        books_combined_final.isna().mean() * 100
    ).round(2)
})

print("\nMissing Values:")
display(
    missing_summary.sort_values(
        "Missing Count",
        ascending=False
    )
)

# Check for empty book categories
empty_categories = books_combined_final["categories"].apply(
    lambda x: isinstance(x, list) and len(x) == 0
).sum()

print("\nEmpty category lists:", empty_categories)

# Check for potentially duplicated review events
duplicate_reviews = books_combined_final.duplicated(
    subset=[
        "user_id", "asin", "timestamp",
        "rating", "text"
    ]
).sum()

print("Potential duplicate review events:", duplicate_reviews)

### Combined Dataset Data Quality Assessment

The merged dataset contains 151,062 observations and 15 variables. All customer identifiers, book identifiers, ratings, timestamps, and book titles are present, indicating that the primary variables needed for the analysis are complete.

Missing information is concentrated in the book metadata. Price is missing for 17,520 observations (11.60%), while main category is missing for only 10 observations. Additionally, 5,894 observations (3.90%) contain empty category lists, demonstrating that conventional null-value checks do not capture all forms of incomplete information.

The duplicate-event assessment also identified 2,012 potentially repeated review records (1.33%) based on selected review attributes. These observations require further examination before determining whether removal is appropriate.

Overall, the dataset is suitable for continued exploratory analysis, although missing metadata, variable formatting, and possible duplicate review events warrant additional investigation before modeling.

In [ ]:
# Identify all rows belonging to repeated review groups
review_keys = [
    "user_id", "asin", "timestamp", "rating", "text"
]

duplicate_mask = books_combined_final.duplicated(
    subset=review_keys,
    keep=False
)

duplicate_candidates = books_combined_final.loc[
    duplicate_mask
].copy()

# Assign a group number to each repeated review event
duplicate_candidates["group_id"] = (
    duplicate_candidates.groupby(
        review_keys,
        dropna=False
    ).ngroup()
)

print("Rows in repeated groups:", len(duplicate_candidates))
print("Repeated review groups:",
      duplicate_candidates["group_id"].nunique())

# Compare potentially duplicated records
display(
    duplicate_candidates[
        [
            "group_id",
            "user_id",
            "asin",
            "parent_asin",
            "rating",
            "review_title",
            "helpful_vote",
            "verified_purchase"
        ]
    ].sort_values("group_id").head(12)
)

### Investigation of Potential Duplicate Reviews

The duplicate assessment identified 3,691 observations belonging to 1,679 repeated review groups, corresponding to 2,012 additional occurrences beyond the first observation in each group.

An inspection of the repeated groups showed that several records share identical customer identifiers, product identifiers, ratings, review titles, helpful-vote counts, and verified-purchase indicators. This suggests that some observations may represent duplicated review events rather than independent customer interactions.

Before removing these observations, the remaining review attributes should be compared to confirm whether the records are genuinely identical. Retaining duplicated reviews could disproportionately influence descriptive statistics and predictive modeling results.

In [ ]:
import json

# Original review fields, excluding book metadata
review_columns = [
    "rating",
    "review_title",
    "text",
    "images",
    "asin",
    "parent_asin",
    "user_id",
    "timestamp",
    "helpful_vote",
    "verified_purchase"
]

review_records = books_combined_final[review_columns].copy()

# Convert nested image data into comparable strings
review_records["images"] = review_records["images"].apply(
    lambda x: json.dumps(x, sort_keys=True, default=str)
)

# Compare complete review records
exact_duplicates = review_records.duplicated().sum()

print("Exact duplicate review records:", exact_duplicates)

print(
    "Potential duplicates differing in other fields:",
    2012 - exact_duplicates
)

### Duplicate Review Validation and Cleaning Decision

The full-record duplicate assessment confirmed 2,012 exact duplicate observations, with no differences across the original review attributes. This verifies that the repeated records identified earlier are identical review events rather than separate observations.

Based on this finding, the 2,012 additional copies will be removed while preserving one occurrence of each review. This prevents duplicated observations from disproportionately influencing descriptive statistics and predictive modeling. The original merged dataset will remain unchanged for reproducibility.

In [ ]:
# Identify exact duplicates from the complete review comparison
exact_duplicate_mask = review_records.duplicated(
    keep="first"
)

# Remove duplicates while preserving the original dataset
books_clean = books_combined_final.loc[
    ~exact_duplicate_mask
].copy()

# Remove the merge-validation indicator
books_clean = books_clean.drop(columns=["_merge"])

# Reset the index
books_clean = books_clean.reset_index(drop=True)

# Verify cleaning results
print("Original rows:", len(books_combined_final))
print("Duplicates removed:", exact_duplicate_mask.sum())
print("Cleaned rows:", len(books_clean))

print("\nCleaned dataset shape:", books_clean.shape)

In [ ]:
# Examine price values after duplicate removal
price_text = books_clean["price"].astype("string").str.strip()

# Convert prices to numeric values for inspection
price_numeric = pd.to_numeric(
    price_text,
    errors="coerce"
)

# Identify missing and invalid prices
missing_prices = books_clean["price"].isna().sum()

invalid_prices = (
    price_text.notna() & price_numeric.isna()
).sum()

print("Missing prices:", missing_prices)

print(
    "Missing price percentage:",
    round(missing_prices / len(books_clean) * 100, 2),
    "%"
)

print("Non-numeric prices:", invalid_prices)

print(
    "Zero or negative prices:",
    (price_numeric <= 0).sum()
)

# Summary statistics for valid numeric prices
print("\nPrice summary:")
display(price_numeric.describe())

# Inspect any values that could not be converted
print("\nExamples of non-numeric prices:")
display(
    books_clean.loc[
        price_text.notna() & price_numeric.isna(),
        ["book_title", "price"]
    ].head(10)
)

### Price Data Quality Assessment

The price assessment identified 17,281 missing observations (11.59%) and an additional 5,465 non-numeric entries. Non-numeric values include unavailable-price indicators and starting-price descriptions, which cannot automatically be interpreted as exact selling prices.

The dataset also contains 9,641 nonpositive numeric prices, with a minimum recorded value of zero. The price distribution appears right-skewed, with a mean of USD 17.62, median of USD 12.35, and maximum of USD 9,653.81.

These findings indicate that price requires additional validation before it can be used reliably in exploratory analysis or predictive modeling. Customer review observations will be retained while invalid, unavailable, and unusually high price values are investigated separately.

In [ ]:
# Reconstruct price values without modifying the dataset
raw_price = books_clean["price"].astype("string").str.strip()

numeric_price = pd.to_numeric(
    raw_price,
    errors="coerce"
)

# Separate zero and negative prices
print("Zero prices:", (numeric_price == 0).sum())
print("Negative prices:", (numeric_price < 0).sum())

# Examine non-numeric price formats
invalid_mask = raw_price.notna() & numeric_price.isna()

print("\nMost common non-numeric price values:")
display(
    raw_price[invalid_mask].value_counts().head(15)
)

# Examine unusually high prices
print("\nTen highest recorded prices:")
price_inspection = books_clean.assign(
    numeric_price=numeric_price
)

display(
    price_inspection[
        ["book_title", "numeric_price"]
    ].nlargest(10, "numeric_price")
)

### Price Format and Outlier Assessment

Further investigation confirmed 9,641 zero-price observations and no negative numeric prices. Non-numeric values include unavailable-price indicators, such as an em-dash, and starting-price descriptions such as "from 19.99." These formats cannot reliably be interpreted as exact book prices.

The dataset also contains unusually high prices, including observations exceeding USD 6,000 and USD 9,000. Although these values may reflect unusual listings or data-quality issues, there is insufficient evidence to classify them as errors.

For subsequent analysis, the original price values will be preserved, while a separate numeric variable will be created. Unavailable or ambiguous prices will remain missing rather than being automatically imputed, and zero and unusually high prices will be flagged for further examination.

In [ ]:
# Preserve original prices
raw_price = books_clean["price"].astype("string").str.strip()

# Convert directly numeric values
numeric_price = pd.to_numeric(raw_price, errors="coerce")

# Classify price formats
books_clean["price_status"] = "other_non_numeric"

books_clean.loc[
    raw_price.isna(), "price_status"
] = "missing"

books_clean.loc[
    raw_price.isin(["—", "–", "-", ""]), "price_status"
] = "unavailable_marker"

books_clean.loc[
    raw_price.str.startswith("from ", na=False), "price_status"
] = "starting_price"

books_clean.loc[
    numeric_price.eq(0).fillna(False), "price_status"
] = "zero"

books_clean.loc[
    numeric_price.gt(0).fillna(False), "price_status"
] = "numeric_positive"

# Create analytical price variable
books_clean["price_for_eda"] = numeric_price.where(
    numeric_price > 0
)

# Review the results
print("Price categories:")
print(books_clean["price_status"].value_counts())

print("\nUsable positive numeric prices:")
print(books_clean["price_for_eda"].notna().sum())

print("\nPrice summary:")
display(books_clean["price_for_eda"].describe())

### Data Preparation Summary

The original Amazon Books dataset contained 29,475,453 customer reviews. To create a manageable analytical dataset, a reproducible customer-based sampling procedure selected 151,062 reviews, which were successfully merged with their corresponding book metadata using `parent_asin`. All sampled reviews matched, with no observations lost or duplicated during the merge.

Following data-quality assessments, 2,012 exact duplicate review records were removed, leaving 149,050 observations. Missing metadata and inconsistent price formats were also evaluated. Rather than removing observations with incomplete price information, the original raw files were preserved, and two analytical variables, `price_status` and `price_for_eda`, were created to classify price formats and retain positive numeric prices. Missing and ambiguous prices were not imputed, and extreme prices were retained for further investigation. This resulted in 116,663 usable price observations.

The resulting dataset is ready for exploratory analysis of customer ratings, review behavior, book characteristics, and relationships relevant to personalized recommendations. Remaining limitations, including missing metadata, sparse customer histories, and potential rating imbalance, will be considered throughout the analysis.

## Exploratory Data Analysis

### 1. Customer Rating Distribution

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Calculate rating counts and percentages
rating_counts = (
    books_clean["rating"]
    .value_counts()
    .sort_index()
)

rating_summary = pd.DataFrame({
    "Count": rating_counts,
    "Percentage": (
        rating_counts / len(books_clean) * 100
    ).round(2)
})

display(rating_summary)

# Calculate positive rating percentage (4-5 stars)
positive_share = (
    books_clean["rating"].ge(4).mean() * 100
)

print(f"Positive rating share: {positive_share:.2f}%")

# Visualize customer rating distribution
rating_colors = [
    "hotpink",
    "mediumvioletred",
    "darkmagenta",
    "orchid",
    "black"
]

plt.figure(figsize=(9, 5))

bars = plt.bar(
    rating_summary.index,
    rating_summary["Count"],
    color=rating_colors
)

# Display percentages above bars
plt.bar_label(
    bars,
    labels=[
        f"{value:.1f}%"
        for value in rating_summary["Percentage"]
    ],
    padding=4
)

plt.xlabel("Customer Rating (Stars)")
plt.ylabel("Number of Reviews")
plt.title("Distribution of Customer Book Ratings")
plt.xticks([1, 2, 3, 4, 5])
plt.ylim(0, rating_summary["Count"].max() * 1.15)

plt.tight_layout()
plt.show()

### Customer Rating Distribution Interpretation

The rating distribution is heavily concentrated toward positive evaluations, with 85.38% of reviews receiving four or five stars and 69.38% receiving five stars. Ratings of one through three stars account for only 14.62% of observations, indicating substantial class imbalance.

This finding has important implications for predictive modeling. If positive ratings are defined as four or five stars, a model predicting every review as positive would achieve 85.38% accuracy without distinguishing customer preferences. Therefore, model evaluation should extend beyond accuracy to include precision, recall, F1-score, and performance against a majority-class baseline.

These findings describe observed reviewer behavior rather than overall purchasing behavior or customer satisfaction across the Amazon Books market.

### 2. Customer Review Behavior

In [ ]:
# Calculate customer review frequency
customer_counts = books_clean["user_id"].value_counts()

print("Unique customers:", len(customer_counts))
print("\nReviews per customer:")
display(customer_counts.describe())

print(
    "\nCustomers with at least 5 reviews:",
    (customer_counts >= 5).sum()
)

print(
    "\nTop 10 customers' share:",
    round(customer_counts.head(10).sum()
          / len(books_clean) * 100, 2),
    "%"
)

# Group customers by number of reviews
frequency_groups = pd.cut(
    customer_counts,
    bins=[0, 1, 2, 4, 9, 19, float("inf")],
    labels=["1", "2", "3–4", "5–9", "10–19", "20+"]
)

frequency_summary = (
    frequency_groups.value_counts().sort_index()
)

display(frequency_summary)

# Visualize customer review frequency
plt.figure(figsize=(9, 5))

bars = plt.bar(
    frequency_summary.index,
    frequency_summary.values,
    color=[
        "hotpink", "mediumvioletred",
        "darkmagenta", "orchid",
        "black", "deeppink"
    ]
)

plt.xlabel("Number of Reviews per Customer")
plt.ylabel("Number of Customers")
plt.title("Distribution of Customer Review Activity")
plt.tight_layout()
plt.show()

### Customer Review Behavior Interpretation

The cleaned dataset contains 51,645 unique customers, with an average of 2.89 reviews per customer. Review activity is highly uneven: approximately 57.98% of customers contributed only one review, while 5,996 customers submitted at least five reviews. The ten most active customers account for 4.76% of all observations.

This distribution highlights an important limitation for personalized recommendation modeling. Most customers have relatively little historical interaction data, making it difficult to identify individual preferences. However, customers with more extensive review histories may provide sufficient information to investigate patterns in rating behavior and book preferences.

These findings suggest that customer interaction frequency should be considered when designing and evaluating predictive models. In particular, model performance may differ between customers with limited histories and those with repeated interactions.

### 3. Book Popularity and Review Frequency

In [ ]:
# Calculate review frequency per book
book_counts = books_clean["parent_asin"].value_counts()

print("Unique books:", len(book_counts))

print("\nReviews per book:")
display(book_counts.describe())

print(
    "\nBooks with only one review:",
    (book_counts == 1).sum()
)

print(
    "\nBooks with at least 5 reviews:",
    (book_counts >= 5).sum()
)

print(
    "\nTop 10 books' share of reviews:",
    round(
        book_counts.head(10).sum() /
        len(books_clean) * 100, 2
    ),
    "%"
)

# Group books by review frequency
book_frequency_groups = pd.cut(
    book_counts,
    bins=[0, 1, 2, 4, 9, float("inf")],
    labels=["1", "2", "3–4", "5–9", "10+"]
)

book_frequency_summary = (
    book_frequency_groups.value_counts().sort_index()
)

display(book_frequency_summary)

# Visualize book review frequency
plt.figure(figsize=(9, 5))

plt.bar(
    book_frequency_summary.index,
    book_frequency_summary.values,
    color=[
        "hotpink",
        "mediumvioletred",
        "darkmagenta",
        "orchid",
        "black"
    ]
)

plt.xlabel("Number of Reviews per Book")
plt.ylabel("Number of Books")
plt.title("Distribution of Book Review Frequency")
plt.tight_layout()
plt.show()

### Book Popularity and Review Frequency Interpretation

The cleaned dataset contains 118,648 unique books, with an average of 1.26 reviews per book. Review frequency is highly sparse, with 104,316 books (87.92%) appearing only once and only 1,656 books receiving at least five reviews. The ten most-reviewed books collectively account for 0.67% of sampled review observations.

This limited interaction history presents a challenge for personalized recommendation models that rely primarily on repeated customer-book interactions. Combined with the sparse customer histories identified earlier, the findings suggest that incorporating book characteristics may provide useful additional information for predicting customer preferences.

Because the analytical dataset was constructed through customer-based sampling, review frequencies reflect activity within the selected sample rather than overall book popularity on Amazon.

### 4. Book Characteristics and Category Analysis

In [ ]:
# Examine categories at the unique-book level
books_unique = books_clean.drop_duplicates(
    subset="parent_asin"
).copy()

print("Unique books:", len(books_unique))

# Examine main category distribution
print("\nMain category distribution:")
display(
    books_unique["main_category"]
    .fillna("Missing")
    .value_counts()
    .head(15)
)

# Check category list completeness
has_categories = books_unique["categories"].apply(
    lambda x: isinstance(x, list) and len(x) > 0
)

print("\nBooks with nonempty category lists:")
print(has_categories.sum())

print("\nBooks with empty or unavailable categories:")
print((~has_categories).sum())

# Inspect examples of category structures
print("\nExamples of book categories:")
display(
    books_unique.loc[
        has_categories,
        ["book_title", "categories"]
    ].head(10)
)

### Book Category Structure Interpretation

The dataset contains 118,648 unique books, of which 113,031 (95.27%) have nonempty category lists. The remaining 5,617 books (4.73%) have empty or unavailable category information.

The `main_category` variable primarily represents broad Amazon classifications, with most books labeled as Books, Buy a Kindle, or Audible Audiobooks. In contrast, the `categories` variable contains hierarchical classifications such as Literature & Fiction, Travel, and Mystery, Thriller & Suspense.

These more detailed categories may provide useful predictors of customer preferences, particularly for books with limited review histories. Further examination is needed to identify a consistent category level for analysis and determine how missing or irregular classifications should be handled.

In [ ]:
# Inspect category hierarchy across unique books
category_lists = books_unique["categories"].apply(
    lambda x: x if isinstance(x, list) else []
)

# Count the number of classification levels
category_depth = category_lists.apply(len)

print("Books with no category levels:",
      (category_depth == 0).sum())

print("Books with one category level:",
      (category_depth == 1).sum())

print("Books with two or more levels:",
      (category_depth >= 2).sum())

# Extract the first and second category levels
first_level = category_lists.apply(
    lambda x: x[0] if len(x) >= 1 else pd.NA
)

second_level = category_lists.apply(
    lambda x: x[1] if len(x) >= 2 else pd.NA
)

print("\nMost common first-level categories:")
display(first_level.value_counts(dropna=False).head(15))

print("\nMost common second-level categories:")
display(second_level.value_counts(dropna=False).head(20))

### Book Category Hierarchy Interpretation

The category hierarchy assessment found that 113,031 books contain at least two classification levels, while 5,617 have no available category information. The first level primarily identifies broad Amazon product classifications, whereas the second level provides more meaningful literary genres and subject areas.

The largest second-level categories include Literature & Fiction (20,273 books), Children's Books (14,026), and Mystery, Thriller & Suspense (11,133). These classifications offer a useful basis for comparing book characteristics and investigating differences in customer ratings.

For subsequent analysis, the second category level will be used to construct a `book_subcategory` variable. Books without available category information will be retained under an Unknown classification rather than excluded.

In [ ]:
# Extract the second-level book category
def get_book_subcategory(categories):
    if isinstance(categories, list) and len(categories) >= 2:
        category = categories[1]

        if isinstance(category, str) and category.strip():
            return category.strip()

    return "Unknown"


books_clean["book_subcategory"] = (
    books_clean["categories"].apply(get_book_subcategory)
)

# Analyze categories at the unique-book level
books_unique = books_clean.drop_duplicates(
    subset="parent_asin"
).copy()

subcategory_counts = (
    books_unique["book_subcategory"].value_counts()
)

# Calculate category counts and percentages
subcategory_summary = pd.DataFrame({
    "Book Count": subcategory_counts,
    "Percentage": (
        subcategory_counts / len(books_unique) * 100
    ).round(2)
})

print("Most common book subcategories:")
display(subcategory_summary.head(15))

# Plot the ten most common known subcategories
top_categories = (
    subcategory_counts
    .drop(index="Unknown", errors="ignore")
    .head(10)
    .sort_values()
)

category_colors = [
    "hotpink",
    "mediumvioletred",
    "darkmagenta",
    "orchid",
    "black"
]

plt.figure(figsize=(10, 6))

plt.barh(
    top_categories.index,
    top_categories.values,
    color=[
        category_colors[i % len(category_colors)]
        for i in range(len(top_categories))
    ]
)

plt.xlabel("Number of Unique Books")
plt.ylabel("Book Subcategory")
plt.title("Top 10 Book Subcategories")
plt.tight_layout()
plt.show()

### Book Subcategory Distribution Interpretation

The subcategory analysis shows that Literature & Fiction represents the largest share of unique books (17.09%), followed by Children's Books (11.82%) and Mystery, Thriller & Suspense (9.38%). Together, these three categories account for 38.29% of the books in the analytical sample. An additional 4.73% of books were classified as Unknown because detailed category information was unavailable.

The distribution demonstrates substantial variation in category representation, suggesting that book subject matter may be useful for examining differences in customer rating behavior. However, these counts reflect the composition of the sampled books rather than overall market demand or sales.

The next step is to compare ratings across subcategories to determine whether customer evaluations vary by book type and whether category information may contribute to predictive modeling.

### 4.1 Customer Ratings by Book Subcategory

In [ ]:
# Identify the 10 largest known book subcategories
top_10_categories = (
    books_unique["book_subcategory"]
    .value_counts()
    .drop(index="Unknown", errors="ignore")
    .head(10)
    .index
)

# Compare ratings across subcategories
category_ratings = (
    books_clean[
        books_clean["book_subcategory"].isin(top_10_categories)
    ]
    .assign(positive_rating=lambda x: x["rating"] >= 4)
    .groupby("book_subcategory")
    .agg(
        review_count=("rating", "size"),
        unique_books=("parent_asin", "nunique"),
        average_rating=("rating", "mean"),
        positive_share=("positive_rating", "mean")
    )
)

# Convert positive share to percentage
category_ratings["positive_pct"] = (
    category_ratings["positive_share"] * 100
)

print("Customer ratings by book subcategory:")

display(
    category_ratings[
        ["review_count", "unique_books",
         "average_rating", "positive_pct"]
    ].sort_values("positive_pct", ascending=False).round(2)
)

# Visualize positive rating percentages
plot_data = category_ratings.sort_values("positive_pct")

plt.figure(figsize=(10, 6))

plt.barh(
    plot_data.index,
    plot_data["positive_pct"],
    color="mediumvioletred"
)

# Overall positive rating baseline
overall_positive = books_clean["rating"].ge(4).mean() * 100

plt.axvline(
    overall_positive,
    color="black",
    linestyle="--",
    label=f"Overall positive rate ({overall_positive:.1f}%)"
)

plt.xlabel("Positive Ratings (4–5 Stars) (%)")
plt.ylabel("Book Subcategory")
plt.title("Positive Rating Share by Book Subcategory")
plt.xlim(0, 100)
plt.legend()
plt.tight_layout()
plt.show()

### Customer Ratings by Book Subcategory Interpretation

Positive-rating percentages vary across the ten largest book subcategories. Christian Books & Bibles has the highest positive-rating share (92.55%), followed by Children's Books (89.64%), while Mystery, Thriller & Suspense (82.73%) and Literature & Fiction (83.62%) fall below the overall positive-rating baseline of 85.38%.

The 9.82-percentage-point difference between the highest- and lowest-rated categories suggests that book subject matter may be associated with customer evaluations. These findings support considering `book_subcategory` as a potential predictive feature. However, the differences are descriptive associations and do not establish that book categories cause higher or lower ratings.

The results also reflect customers who submitted reviews, rather than all Amazon customers or actual purchase behavior.

### 4.2 Book Price Distribution

In [ ]:
# Analyze prices at the unique-book level
books_unique = books_clean.drop_duplicates(
    subset="parent_asin"
).copy()

book_prices = (
    books_unique["price_for_eda"]
    .dropna()
    .astype(float)
)

# Price coverage and descriptive statistics
print("Unique books:", len(books_unique))
print("Books with usable prices:", len(book_prices))

print(
    "Price coverage:",
    round(len(book_prices) / len(books_unique) * 100, 2),
    "%"
)

print("\nPrice summary:")
display(
    book_prices.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

# Limit the visualization to the 99th percentile
price_limit = book_prices.quantile(0.99)

plot_prices = book_prices[
    book_prices <= price_limit
]

print(f"\n99th percentile: ${price_limit:.2f}")
print("Prices excluded from chart only:",
      len(book_prices) - len(plot_prices))

# Visualize book prices
plt.figure(figsize=(9, 5))

plt.hist(
    plot_prices,
    bins=40,
    color="mediumvioletred",
    edgecolor="white"
)

plt.axvline(
    book_prices.median(),
    color="black",
    linestyle="--",
    label=f"Median: ${book_prices.median():.2f}"
)

plt.xlabel("Book Listing Price ($)")
plt.ylabel("Number of Unique Books")
plt.title("Distribution of Book Prices (Up to 99th Percentile)")
plt.legend()
plt.tight_layout()
plt.show()

### Book Price Distribution Interpretation

Of the 118,648 unique books in the analytical sample, 93,787 (79.05%) have usable positive numeric prices. The price distribution is strongly right-skewed, with a median of USD 13.79 and a mean of $20.26. Approximately 75% of books with usable prices are listed at or below USD 19.99, while a small number of unusually expensive listings extend the distribution considerably.

To improve visualization, the histogram was restricted to the 99th percentile ($126.42), excluding 938 observations from the chart only. All prices were retained in the analytical dataset.

These findings demonstrate considerable variation in book listing prices, suggesting that price may be worth examining alongside customer ratings. However, the recorded prices are product metadata rather than confirmed transaction prices and may not reflect prices at the time reviews were submitted.

### 4.3 Customer Ratings by Book Price

In [ ]:
# Select reviews with usable book prices
priced_reviews = books_clean.loc[
    books_clean["price_for_eda"].notna(),
    ["rating", "parent_asin", "price_for_eda"]
].copy()

# Create interpretable price ranges
priced_reviews["price_band"] = pd.cut(
    priced_reviews["price_for_eda"].astype(float),
    bins=[0, 10, 20, 40, float("inf")],
    labels=[
        "Under $10",
        "$10–<$20",
        "$20–<$40",
        "$40+"
    ],
    right=False
)

# Identify positive customer ratings
priced_reviews["positive_rating"] = (
    priced_reviews["rating"] >= 4
)

# Compare ratings across price ranges
price_rating_summary = (
    priced_reviews
    .groupby("price_band", observed=True)
    .agg(
        review_count=("rating", "size"),
        unique_books=("parent_asin", "nunique"),
        average_rating=("rating", "mean"),
        positive_share=("positive_rating", "mean")
    )
)

price_rating_summary["positive_pct"] = (
    price_rating_summary["positive_share"] * 100
)

print("Customer ratings by book price range:")

display(
    price_rating_summary[
        ["review_count", "unique_books",
         "average_rating", "positive_pct"]
    ].round(2)
)

# Baseline among reviews with usable prices
priced_positive_baseline = (
    priced_reviews["positive_rating"].mean() * 100
)

# Visualize positive rating percentages
plt.figure(figsize=(9, 5))

plt.barh(
    price_rating_summary.index.astype(str),
    price_rating_summary["positive_pct"],
    color="mediumvioletred"
)

plt.axvline(
    priced_positive_baseline,
    color="black",
    linestyle="--",
    label=f"Priced-review baseline ({priced_positive_baseline:.1f}%)"
)

plt.xlabel("Positive Ratings (4–5 Stars) (%)")
plt.ylabel("Book Price Range")
plt.title("Positive Rating Share by Book Price Range")
plt.xlim(0, 100)
plt.legend()
plt.tight_layout()
plt.show()

### Customer Ratings by Book Price Interpretation

Positive-rating percentages are relatively consistent across book price ranges, varying from 84.66% for books under USD 10 to 86.17% for books priced between USD 20 and USD 40. The highest-priced group (USD 40 and above) has a positive-rating share of 85.37%, indicating no consistent increase in favorable ratings as price rises.

The difference of 1.51 percentage points between the highest- and lowest-rated price groups is substantially smaller than the variation observed across book subcategories. This suggests that listing price may provide less predictive information about customer ratings than book subject matter, although its contribution must be evaluated through modeling.

These findings represent descriptive associations rather than causal relationships, and the available prices may not reflect amounts paid by customers at the time of their reviews.

### 5. Review Activity Over Time

In [ ]:
# Extract review year
books_clean["review_year"] = (
    pd.to_datetime(books_clean["timestamp"], errors="coerce")
    .dt.year
)

# Summarize annual review activity
yearly_reviews = (
    books_clean
    .assign(positive_rating=books_clean["rating"] >= 4)
    .groupby("review_year")
    .agg(
        review_count=("rating", "size"),
        unique_customers=("user_id", "nunique"),
        average_rating=("rating", "mean"),
        positive_share=("positive_rating", "mean")
    )
)

# Convert positive share to percentage
yearly_reviews["positive_pct"] = (
    yearly_reviews["positive_share"] * 100
)

print("Review activity by year:")

display(
    yearly_reviews[
        ["review_count", "unique_customers",
         "average_rating", "positive_pct"]
    ].round(2)
)

# Visualize annual review activity
plt.figure(figsize=(11, 5))

plt.bar(
    yearly_reviews.index.astype(int),
    yearly_reviews["review_count"],
    color="mediumvioletred"
)

plt.xlabel("Review Year")
plt.ylabel("Number of Reviews")
plt.title("Annual Amazon Books Review Activity")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Review Activity Over Time Interpretation

Annual review activity increased substantially through the mid-2010s, reaching a peak of 15,618 observations in 2015 before declining in subsequent years. Positive-rating percentages also varied over time, increasing from 75.44% in 2004 to 87.50% in 2019, followed by a modest decline to 85.12% in 2022.

These patterns suggest that review activity and rating behavior may differ across historical periods. However, the results reflect the selected customer-based sample and should not be interpreted as direct measures of Amazon's overall sales or customer activity. Additionally, the incomplete coverage of 2023 limits comparisons with full calendar years.

The observed temporal variation suggests that review year may be useful for further exploratory analysis. It also supports considering chronological data splitting during predictive modeling to evaluate performance on later observations and reduce the risk of temporal data leakage.

### 5.1 Positive Rating Trends Over Time

In [ ]:
# Focus on years with sufficient review observations
# Exclude incomplete 2023 for historical comparison
yearly_trends = yearly_reviews.loc[
    (yearly_reviews.index <= 2022)
    & (yearly_reviews["review_count"] >= 500)
].copy()

# Calculate baseline for the selected years
historical_baseline = (
    books_clean.loc[
        books_clean["review_year"].isin(yearly_trends.index),
        "rating"
    ].ge(4).mean() * 100
)

# Visualize positive-rating trends
plt.figure(figsize=(11, 5))

plt.plot(
    yearly_trends.index,
    yearly_trends["positive_pct"],
    color="mediumvioletred",
    marker="o",
    linewidth=2
)

plt.axhline(
    historical_baseline,
    color="black",
    linestyle="--",
    label=f"Historical baseline ({historical_baseline:.1f}%)"
)

plt.xlabel("Review Year")
plt.ylabel("Positive Ratings (4–5 Stars) (%)")
plt.title("Positive Book Rating Trends Over Time")
plt.xticks(rotation=45)
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### Positive Rating Trends Over Time Interpretation

The positive-rating trend shows substantial variation across the historical period, increasing from a low of 75.44% in 2004 to 87.50% in 2019. Positive ratings remained relatively stable around 87% during 2014–2019 before declining to 84.63% in 2021 and recovering slightly to 85.12% in 2022.

These findings suggest that the distribution of customer ratings may change over time, which has implications for predictive modeling. A chronological training and testing approach may help evaluate how well models generalize to later observations rather than relying exclusively on randomly selected records.

However, the observed trends are descriptive and may reflect changes in reviewer composition, book characteristics, or data coverage rather than changes in overall customer satisfaction.

### 6. Relationships Between Book Characteristics

In [ ]:
# Examine price and category relationships together
top_categories = (
    books_unique["book_subcategory"]
    .value_counts()
    .drop(index="Unknown", errors="ignore")
    .head(10)
    .index
)

feature_data = books_clean.loc[
    books_clean["book_subcategory"].isin(top_categories)
    & books_clean["price_for_eda"].notna(),
    ["book_subcategory", "price_for_eda", "rating"]
].copy()

# Create price bands
price_labels = [
    "Under $10", "$10–<$20", "$20–<$40", "$40+"
]

feature_data["price_band"] = pd.cut(
    feature_data["price_for_eda"].astype(float),
    bins=[0, 10, 20, 40, float("inf")],
    labels=price_labels,
    right=False
)

feature_data["positive_rating"] = (
    feature_data["rating"] >= 4
)

# Calculate positive rating percentages and sample sizes
relationship_summary = (
    feature_data
    .groupby(["book_subcategory", "price_band"], observed=True)
    .agg(
        review_count=("rating", "size"),
        positive_pct=("positive_rating", "mean")
    )
)

relationship_summary["positive_pct"] *= 100

# Reshape results for visualization
positive_matrix = (
    relationship_summary["positive_pct"]
    .unstack("price_band")
    .reindex(index=top_categories, columns=price_labels)
)

count_matrix = (
    relationship_summary["review_count"]
    .unstack("price_band")
    .reindex(index=top_categories, columns=price_labels)
)

# Hide groups with fewer than 100 reviews
positive_matrix = positive_matrix.where(
    count_matrix >= 100
)

display(positive_matrix.round(2))

# Create heat map
plt.figure(figsize=(11, 7))

plt.imshow(
    positive_matrix.astype(float),
    aspect="auto",
    cmap="RdPu",
    vmin=75,
    vmax=100
)

plt.colorbar(label="Positive Ratings (%)")

plt.xticks(
    range(len(positive_matrix.columns)),
    positive_matrix.columns
)

plt.yticks(
    range(len(positive_matrix.index)),
    positive_matrix.index
)

plt.xlabel("Book Price Range")
plt.ylabel("Book Subcategory")
plt.title("Positive Rating Share by Book Category and Price")

plt.tight_layout()
plt.show()

### Relationships Between Book Characteristics Interpretation

The heat map shows that positive-rating patterns vary across book subcategories and price ranges. Christian Books & Bibles consistently exhibits relatively high positive-rating percentages, while Mystery, Thriller & Suspense and Literature & Fiction generally show lower percentages. Price-related patterns also differ across categories, with no consistent relationship between higher listing prices and more favorable customer ratings.

These findings suggest that book subcategory may be more informative than price alone when examining customer evaluations. They also indicate that the relationship between price and ratings may depend on book category. However, these descriptive associations do not establish causality or predictive importance.

The results support evaluating category and price as potential modeling features, while accounting for reviewer differences, incomplete pricing information, and the historical availability of product metadata.

In [ ]:
# Display positive-rating percentages and review counts
# for each category-price combination

category_price_details = relationship_summary[
    ["review_count", "positive_pct"]
].copy()

# Retain combinations with at least 100 reviews
category_price_details = category_price_details.loc[
    category_price_details["review_count"] >= 100
]

print("Category and price relationship details:")

display(
    category_price_details.round(2)
)

# Examine variation in positive ratings across price
# bands within each book subcategory
category_price_spread = (
    positive_matrix.max(axis=1)
    - positive_matrix.min(axis=1)
)

print("\nPositive-rating variation across price bands:")
display(
    category_price_spread
    .sort_values(ascending=False)
    .round(2)
)

### Category and Price Relationship Validation

The detailed category-price comparison confirms that positive-rating patterns vary across book subcategories. History exhibits the greatest variation across price bands (7.61 percentage points), followed by Mystery, Thriller & Suspense (6.05 points). In contrast, Christian Books & Bibles (1.50 points) and Literature & Fiction (1.66 points) demonstrate relatively stable positive-rating percentages.

These results suggest that the association between listing price and customer ratings may depend on book category rather than following a consistent pattern across all books. However, some category-price combinations contain relatively few reviews, and differences may reflect reviewer characteristics, product composition, or other unobserved factors.

Overall, the findings support considering book subcategory and price as potential predictive features while emphasizing that descriptive relationships do not necessarily establish predictive importance or causality.

### 7. Modeling Readiness and Feature Availability

In [ ]:
# Assess availability of potential modeling features
model_audit = books_clean.copy()

# Provisional classification target
model_audit["positive_rating"] = (
    model_audit["rating"] >= 4
)

# Flag available book characteristics
model_audit["price_available"] = (
    model_audit["price_for_eda"].notna()
)

model_audit["category_available"] = (
    model_audit["book_subcategory"] != "Unknown"
)

# Compare rating outcomes by feature availability
for feature in ["price_available", "category_available"]:

    print(f"\nPositive ratings by {feature}:")

    summary = (
        model_audit
        .groupby(feature)
        .agg(
            review_count=("rating", "size"),
            positive_pct=("positive_rating", "mean")
        )
    )

    summary["positive_pct"] *= 100

    display(summary.round(2))

### Modeling Readiness and Feature Availability Interpretation

The feature-availability assessment found relatively small differences in positive-rating percentages between observations with and without usable book characteristics. Reviews with available prices had an 85.32% positive-rating share, compared with 85.57% for those without usable prices. Similarly, reviews with available categories had an 85.34% positive-rating share, compared with 86.21% for those without category information.

These results suggest that missing metadata is not associated with large differences in overall positive-rating percentages. However, the findings do not establish that missingness is random or that excluding incomplete observations would introduce no bias.

Retaining incomplete observations is therefore appropriate for the modeling handoff. Missing-price treatment and categorical encoding should be determined within the modeling pipeline, with preprocessing fitted on training data to avoid data leakage.

### 8. Modeling Dataset Preparation

In [ ]:
# Select fields for the modeling handoff
handoff_columns = [
    "user_id",
    "parent_asin",
    "timestamp",
    "main_category",
    "book_subcategory",
    "price_for_eda",
    "price_status"
]

model_handoff = books_clean[handoff_columns].copy()

# Define provisional binary classification target
# 1 = positive rating (4-5 stars)
# 0 = nonpositive rating (1-3 stars)
model_handoff["positive_rating"] = (
    books_clean["rating"] >= 4
).astype(int)

# Verify the handoff
print("Modeling dataset shape:", model_handoff.shape)

print("\nTarget distribution:")
print(model_handoff["positive_rating"].value_counts())

print("\nMissing values:")
print(model_handoff.isna().sum())

# Export candidate dataset
model_handoff.to_csv(
    "books_modeling_handoff_candidate.csv",
    index=False
)

print("\nCandidate modeling dataset saved!")

### Modeling Dataset Handoff Validation

The modeling handoff dataset contains 149,050 observations and eight variables, including customer and book identifiers, review timestamps, book characteristics, and a provisional binary classification target. The target identifies 127,253 positive ratings (85.38%) and 21,797 nonpositive ratings (14.62%), confirming substantial class imbalance.

The dataset retains all cleaned review observations, including 32,387 records without usable numeric prices and 10 records with missing main-category information. Missing-value treatment, categorical encoding, and feature selection will be addressed during modeling.

This dataset serves as a reproducible candidate handoff for classification analysis. However, the team must establish which predictors would be available before a customer submits a review and select an appropriate validation strategy to prevent data leakage.

## Overall EDA Findings and Modeling Recommendations

### Key Findings

Exploratory data analysis was conducted on 149,050 cleaned Amazon Books reviews, representing 51,645 customers and 118,648 unique books. The analysis identified several patterns relevant to predicting customer ratings and supporting personalized book recommendations.

**Customer ratings and interaction patterns:** Customer evaluations are heavily skewed toward positive ratings, with 85.38% receiving four or five stars. Additionally, approximately 58% of customers contributed only one review, while 87.92% of books received only one review within the sample. This limited interaction history presents challenges for identifying individual customer preferences.

**Book characteristics:** Positive-rating percentages varied meaningfully across book subcategories. Christian Books & Bibles had the highest positive-rating share among the ten largest categories (92.55%), while Mystery, Thriller & Suspense had the lowest (82.73%). In contrast, positive-rating percentages across price bands ranged from 84.66% to 86.17%, suggesting a weaker overall association between listing price and ratings. Further analysis indicated that price-related patterns differed across book categories.

**Temporal patterns and data quality:** Positive-rating percentages varied across historical periods, suggesting potential changes in rating distributions over time. Missing and inconsistent price information also required additional processing. Rather than discarding incomplete observations, the dataset retained missing-value indicators and preserved available customer interactions.

### Modeling Recommendations

The findings support evaluating classification methods, including logistic regression and random forest, to predict whether an observed customer review receives a positive rating (4–5 stars).

Because 85.38% of reviews are positive, models should be evaluated against a majority-class baseline rather than relying exclusively on accuracy. Precision, recall, F1-score, and confusion matrices will be particularly important for assessing performance on nonpositive ratings.

Book subcategory and price should be evaluated as potential predictors, with appropriate categorical encoding and missing-value treatment. Preprocessing should be fitted using training data only, while chronological and customer-aware validation strategies should be considered to minimize data leakage. Features derived from completed reviews or future customer activity should not be used for predictions intended to occur before a review is submitted.

### Business Implications and Limitations

The EDA suggests that book characteristics may help distinguish patterns in customer evaluations. However, limited customer and book interaction histories present challenges for developing individualized recommendations.

The current modeling dataset primarily captures book characteristics rather than established customer preferences. Stronger personalization would require additional features derived from customers' prior interactions. Furthermore, the available data represent submitted reviews rather than complete purchasing behavior, so findings should not be interpreted as direct evidence of sales performance.

Overall, the analysis provides a foundation for comparing predictive models and identifying whether book characteristics contribute useful information beyond baseline rating patterns.

In [ ]:
import numpy as np, pandas as pd, time, warnings
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score, balanced_accuracy_score,
                             f1_score, precision_score, recall_score, roc_curve, precision_recall_curve,
                             confusion_matrix, log_loss, brier_score_loss)
warnings.filterwarnings("ignore")
RANDOM_STATE = 42

df = model_handoff

**Data Subsets**
First we will split our data into training, validation, and test data sets. We will use different date ranges in order to make our subset divisions.

In [ ]:
n = len(df)
train_end, val_end = int(n * 0.70), int(n * 0.85)
df["split"] = np.where(df.index < train_end, "train", np.where(df.index < val_end, "val", "test"))

p_bar = df.loc[df.split == "train", "positive_rating"].mean()
M = 5

def add_history(frame, key, prefix):
    g = frame.groupby(key, sort=False)["positive_rating"]
    prior_n = g.cumcount()                                   # reviews strictly before this one
    prior_pos = g.cumsum() - frame["positive_rating"]        # positives strictly before this one
    frame[f"{prefix}_prior_n"] = prior_n
    frame[f"{prefix}_prior_rate"] = (prior_pos + M * p_bar) / (prior_n + M)

add_history(df, "user_id", "user")
add_history(df, "parent_asin", "book")

df["price_missing"] = df["price_for_eda"].isna().astype(int)
df["log_price"] = np.log1p(df["price_for_eda"])
df["year"] = df["timestamp"].dt.year
df["main_category"] = df["main_category"].fillna("Unknown")

# Sanity check: no feature uses the row's own target
chk = df[df.user_id == df.user_id.value_counts().index[3]].head(4)[["timestamp","positive_rating","user_prior_n","user_prior_rate"]]
print(chk)

for s in ["train", "val", "test"]:
    d = df[df.split == s]
    print(f"{s:5s} n={len(d):6d}  {d.timestamp.min().date()} -> {d.timestamp.max().date()}  positive rate={d.positive_rating.mean():.4f}")

In [ ]:
META_NUM = ["log_price", "price_missing", "year"]
META_CAT = ["price_status", "main_category", "book_subcategory"]
HIST_NUM = ["user_prior_n", "user_prior_rate", "book_prior_n", "book_prior_rate"]

# user_prior_n / book_prior_n are heavy-tailed -> log transform
for c in ["user_prior_n", "book_prior_n"]:
    df[c] = np.log1p(df[c])

ALL_FEATURES = META_NUM + META_CAT + HIST_NUM
tr, va, te = (df[df.split == s] for s in ["train", "val", "test"])
y_tr, y_va, y_te = tr.positive_rating.values, va.positive_rating.values, te.positive_rating.values

def make_preprocessor(num_cols, cat_cols):
    parts = []
    if num_cols:
        parts.append(("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                                       ("sc", StandardScaler())]), num_cols))
    if cat_cols:
        parts.append(("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=200), cat_cols))
    return ColumnTransformer(parts)

def evaluate(y, p, thr=0.5):
    pred = (p >= thr).astype(int)
    return {"ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p),
            "Log loss": log_loss(y, np.clip(p, 1e-6, 1-1e-6)),
            "Accuracy": accuracy_score(y, pred), "Balanced acc.": balanced_accuracy_score(y, pred),
            "F1 (negative)": f1_score(y, pred, pos_label=0, zero_division=0),
            "Recall (negative)": recall_score(y, pred, pos_label=0, zero_division=0)}

def best_threshold(y, p):
    # threshold maximising balanced accuracy on VALIDATION data
    grid = np.linspace(0.05, 0.95, 91)
    return grid[np.argmax([balanced_accuracy_score(y, (p >= t).astype(int)) for t in grid])]

## Baseline models

Because ~85% of ratings are positive, **accuracy alone is misleading**: always predicting "positive" scores ~85% while identifying no unhappy customers. So we also track ROC-AUC, PR-AUC, log loss, and balanced accuracy / F1 on the minority (not-positive) class.

Baselines:
- **Majority class:** always predict positive.
- **Prior probability:** predict the training positive rate for everyone (zero ranking ability, AUC = 0.5).
- **Simple heuristic:** predict using the customer's prior positive rate alone (a "customer history only" rule).

In [ ]:
rows = {}
maj = DummyClassifier(strategy="most_frequent").fit(tr[ALL_FEATURES], y_tr)
rows["Baseline: always positive"] = evaluate(y_te, maj.predict_proba(te[ALL_FEATURES])[:, 1], 0.5)

prior = DummyClassifier(strategy="prior").fit(tr[ALL_FEATURES], y_tr)
rows["Baseline: training prior"] = evaluate(y_te, prior.predict_proba(te[ALL_FEATURES])[:, 1], 0.5)

thr_h = best_threshold(y_va, va["user_prior_rate"].values)
rows["Baseline: customer history rule"] = evaluate(y_te, te["user_prior_rate"].values, thr_h)

baseline_df = pd.DataFrame(rows).T.round(4)
baseline_df

## Candidate models

Four different models are tested to see their predictive power and compare them to the baseline model, logistic regression, decision tree, random forest, and k-nearest neighbors. Because of the high baseline accuracy for an always positive model and the strategic interest in negative responses, we are most interested in testing ROC AUC, F1 negative and Recall negative values.

In [ ]:
from itertools import product
pre = lambda: make_preprocessor(META_NUM + HIST_NUM, META_CAT)

grids = {
    "Logistic Regression": (lambda **k: LogisticRegression(max_iter=2000, **k),
                            [dict(C=c) for c in [0.01, 0.1, 1, 10]]),
    "Decision Tree": (lambda **k: DecisionTreeClassifier(random_state=RANDOM_STATE, **k),
                      [dict(max_depth=d, min_samples_leaf=l) for d, l in product([3, 5, 8, 12], [25, 100, 300])]),
    "Random Forest": (lambda **k: RandomForestClassifier(n_estimators=150, max_features="sqrt", n_jobs=-1,
                                                          random_state=RANDOM_STATE, **k),
                      [dict(max_depth=d, min_samples_leaf=l) for d, l in product([8, 14], [10, 50])]),
    "k-Nearest Neighbors": (lambda **k: KNeighborsClassifier(n_jobs=-1, **k),
                            [dict(n_neighbors=k_) for k_ in [25, 75, 200]]),
}

tuning_log, fitted = [], {}
for name, (ctor, grid) in grids.items():
    best = None
    for params in grid:
        t0 = time.time()
        pipe = Pipeline([("pre", pre()), ("clf", ctor(**params))]).fit(tr[ALL_FEATURES], y_tr)
        auc = roc_auc_score(y_va, pipe.predict_proba(va[ALL_FEATURES])[:, 1])
        tuning_log.append({"Model": name, "Params": str(params), "Val ROC-AUC": auc, "Seconds": time.time() - t0})
        if best is None or auc > best[0]:
            best = (auc, params, pipe)
    fitted[name] = best
    print(f"{name:22s} best params={best[1]}  val AUC={best[0]:.4f}")

tuning_df = pd.DataFrame(tuning_log)
tuning_df.round(4)

Comparison of all 4 predictive models to the baseline models.

In [ ]:
results, probs, thresholds = {}, {}, {}
for name, (auc, params, pipe) in fitted.items():
    p_va = pipe.predict_proba(va[ALL_FEATURES])[:, 1]
    thr = best_threshold(y_va, p_va)
    p_te = pipe.predict_proba(te[ALL_FEATURES])[:, 1]
    results[name] = evaluate(y_te, p_te, thr)
    probs[name], thresholds[name] = p_te, thr

results_df = pd.DataFrame(results).T
results_df.insert(0, "Threshold", pd.Series(thresholds))
comparison = pd.concat([baseline_df.assign(Threshold=[0.5, 0.5, thr_h]), results_df]).round(4)
comparison.sort_values("ROC-AUC", ascending=False)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
for name, p in probs.items():
    fpr, tpr, _ = roc_curve(y_te, p); ax[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_te, p):.3f})")
    pr, rc, _ = precision_recall_curve(1 - y_te, 1 - p); ax[1].plot(rc, pr, label=name)
ax[0].plot([0, 1], [0, 1], "k--", lw=1); ax[0].set(title="ROC curves (test)", xlabel="False positive rate", ylabel="True positive rate"); ax[0].legend()
ax[1].axhline(1 - y_te.mean(), color="k", ls="--", lw=1)
ax[1].set(title="Precision-recall for NOT-positive class (test)", xlabel="Recall", ylabel="Precision"); ax[1].legend()

res = results_df["ROC-AUC"].sort_values()
ax[2].barh(res.index, res.values, color="#4C72B0"); ax[2].axvline(0.5, color="k", ls="--", lw=1)
ax[2].set(title="Test ROC-AUC", xlim=(0.45, max(0.7, res.max() + 0.03)))
for i, v in enumerate(res.values): ax[2].text(v + 0.002, i, f"{v:.3f}", va="center")
plt.tight_layout(); plt.show()

In [ ]:
best_name = results_df["ROC-AUC"].idxmax()
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for a, (name, p) in zip(ax, probs.items()):
    cm = confusion_matrix(y_te, (p >= thresholds[name]).astype(int))
    a.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2): a.text(j, i, f"{cm[i, j]:,}", ha="center", va="center")
    a.set(title=f"{name}\nthr={thresholds[name]:.2f}", xticks=[0, 1], yticks=[0, 1],
          xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
plt.tight_layout(); plt.show()
print("Best model by test ROC-AUC:", best_name)

## What drives prediction?

**Feature-group ablation:** refit the tuned models using (a) book/metadata features only, (b) customer/book history only, and (c) everything.

In [ ]:
sets = {"Metadata only": (META_NUM, META_CAT),
        "History only": (HIST_NUM, []),
        "All features": (META_NUM + HIST_NUM, META_CAT)}
ctors = {k: v[0] for k, v in grids.items()}
abl = []
for name in ["Logistic Regression", "Decision Tree", "Random Forest"]:
    params = fitted[name][1]
    for sname, (nc, cc) in sets.items():
        pipe = Pipeline([("pre", make_preprocessor(nc, cc)), ("clf", ctors[name](**params))]).fit(tr[nc + cc], y_tr)
        abl.append({"Model": name, "Features": sname,
                    "Test ROC-AUC": roc_auc_score(y_te, pipe.predict_proba(te[nc + cc])[:, 1])})
abl_df = pd.DataFrame(abl).pivot(index="Model", columns="Features", values="Test ROC-AUC")[list(sets)].round(4)
abl_df

In [ ]:
pipe = fitted[best_name][2]
pi = permutation_importance(pipe, te[ALL_FEATURES], y_te, scoring="roc_auc", n_repeats=5, random_state=RANDOM_STATE)
imp = pd.Series(pi.importances_mean, index=ALL_FEATURES).sort_values()
ax = imp.plot.barh(figsize=(7, 4.5), color="#55A868", title=f"Permutation importance - {best_name}")
ax.set_xlabel("Decrease in test ROC-AUC"); plt.tight_layout(); plt.show()

if best_name == "Logistic Regression":
    names = pipe.named_steps["pre"].get_feature_names_out()
    coef = pd.Series(pipe.named_steps["clf"].coef_[0], index=names).sort_values()
    display(pd.concat([coef.head(8), coef.tail(8)]).round(3).to_frame("coefficient"))

## Are the model differences real? And what about customers with no history?

Because prior customer ratings is by far our strongest predictor, does that mean that the model is not useful for customers without any prior ratings history? If so, what is the predictive performance difference between customers with history and without? Bootstrap 95% intervals for test ROC-AUC, plus performance split by whether the customer had any earlier review (**cold start** vs **has history**).

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
B = 200
idx_all = [rng.integers(0, len(y_te), len(y_te)) for _ in range(B)]
boot = {}
for name, p in list(probs.items()) + [("Customer history rule", te["user_prior_rate"].values)]:
    aucs = [roc_auc_score(y_te[i], p[i]) for i in idx_all]
    boot[name] = (roc_auc_score(y_te, p), np.percentile(aucs, 2.5), np.percentile(aucs, 97.5))
boot_df = pd.DataFrame(boot, index=["ROC-AUC", "CI low", "CI high"]).T.round(4)
display(boot_df)


diff = [roc_auc_score(y_te[i], probs[best_name][i]) - roc_auc_score(y_te[i], te["user_prior_rate"].values[i]) for i in idx_all]
print(f"{best_name} minus customer-history rule: mean {np.mean(diff):+.4f}, 95% CI [{np.percentile(diff, 2.5):+.4f}, {np.percentile(diff, 97.5):+.4f}]")

In [ ]:
has_hist = (te["user_prior_n"].values > 0)
print(f"Test reviews from customers with prior history: {has_hist.mean():.1%}")
rows = []
for name, p in probs.items():
    for label, m in [("Cold start (no prior reviews)", ~has_hist), ("Has prior history", has_hist)]:
        rows.append({"Model": name, "Segment": label, "n": int(m.sum()),
                     "Positive rate": y_te[m].mean(), "ROC-AUC": roc_auc_score(y_te[m], p[m])})
pd.DataFrame(rows).pivot(index="Model", columns="Segment", values="ROC-AUC").round(4).join(
    pd.DataFrame(rows).groupby("Segment")[["n", "Positive rate"]].first().T.rename(index={"n": "n (segment size)"}), how="outer") if False else pd.DataFrame(rows).pivot(index="Model", columns="Segment", values="ROC-AUC").round(4)

## Further Examination of the Logistic Regression Model

Because all of our models perform nearly identically, we can default to the most efficient model timewise and memory wise, the logistic regression.

In [ ]:

lr_pipe = fitted["Logistic Regression"][2]


names = lr_pipe.named_steps["pre"].get_feature_names_out()
coefs = lr_pipe.named_steps["clf"].coef_[0]

coef_df = pd.DataFrame({
    "feature": names,
    "coefficient": coefs,
    "odds_ratio": np.exp(coefs),
}).sort_values("coefficient", ascending=False)


num_df = coef_df[coef_df["feature"].str.startswith("num__")]
print("Numeric features (per 1 standard deviation increase):")
display(num_df.round(3))

print("\\nTop 10 categorical coefficients (push toward positive):")
display(coef_df[coef_df["feature"].str.startswith("cat__")].head(10).round(3))
print("\\nBottom 10 categorical coefficients (push toward not-positive):")
display(coef_df[coef_df["feature"].str.startswith("cat__")].tail(10).round(3))


pi_lr = permutation_importance(lr_pipe, te[ALL_FEATURES], y_te, scoring="roc_auc",
                               n_repeats=5, random_state=RANDOM_STATE)
imp_lr = pd.DataFrame({"importance_mean": pi_lr.importances_mean,
                       "importance_std": pi_lr.importances_std},
                      index=ALL_FEATURES).sort_values("importance_mean", ascending=False)
print("\\nPermutation importance (drop in test ROC-AUC):")
display(imp_lr.round(4))

imp_lr["importance_mean"].sort_values().plot.barh(
    figsize=(7, 4.5), color="#4C72B0", title="Permutation importance - Logistic Regression")
plt.xlabel("Decrease in test ROC-AUC"); plt.tight_layout(); plt.show()

## Conclusions

**Baselines.** Always predicting "positive" gives 84.8% test accuracy but ROC-AUC 0.50 and catches no unhappy ratings. A one-feature rule based on the customer's prior positive rate reaches ROC-AUC 0.697. However, this does not give any insight into which books customers will not prefer.

**Model comparison (test ROC-AUC, 95% bootstrap CI).** Random forest 0.702 [0.694, 0.711], decision tree 0.702 [0.694, 0.710], logistic regression 0.701 [0.693, 0.710], k-NN 0.701 [0.692, 0.710]. The intervals overlap almost completely, so **the four models are statistically indistinguishable**. Random forest beats the customer-history rule by only +0.005 (95% CI +0.000 to +0.011). Given the similar effectiveness of each model, we should choose to use the logistic regression as it is the simplest and easiest to work with.

**What predicts positive ratings.** Customer rating history is the dominant signal. Book metadata alone (price, category, subcategory, year) gives ROC-AUC of only about 0.53 to 0.55, history alone gives about 0.69 to 0.70, and combining them adds almost nothing. Specifically, the highest predictive signaal is the customer history of positive ratings. Customer's who have rated books highly in the past are much more likely to rate books highly in the future.

**Cold start.** About 65% of test reviews come from customers with prior reviews (ROC-AUC about 0.75). The remaining 35% have no history, and there performance falls to about 0.55 to 0.58, close to chance. k-NN and random forest are slightly better in the cold-start segment. This proportion of customers may stay steady or shrink as customers provide more reviews, and as new customers join the dataset.

**Recommendation.** Use logistic regression or a shallow decision tree as the working model: they perform as well as the more complex models and are easy to explain. To improve meaningfully, add richer signals (e.g., review text, book average rating, author history, gradient boosting) rather than tuning these four further.

**Limitations.** Data is a sample of Amazon reviews, so history is incomplete. Thresholds were tuned for balanced accuracy, and other thresholds suit other business costs. Results come from a single chronological split. Customer reviews are also a biased dataset because, much like survey results, they will only show behaviors related to people who are likely to submit reviews.

**GitHub** https://github.com/GuhanShastri/ADS505